# 04_parameter_estimation_realistic

This notebook prepares all model inputs required for the realistic scenario in Part 2.

**Main tasks**
1. Load the shared cleaned datasets and the ideal-model outputs.
2. Build realistic expansion parameters for existing facilities.
3. Build candidate-site construction options for new facilities.
4. Build minimum-distance conflict tables for candidate sites.
5. Export all realistic-model input tables.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)

In [3]:
def find_project_root():
    """Return a reasonable project root based on the current working directory."""
    cwd = Path.cwd()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for p in candidates:
        if (p / "data").exists():
            return p
    return cwd

PROJECT_ROOT = find_project_root()
SHARED_DIR = PROJECT_ROOT / "data" / "processed" / "shared"
IDEAL_DIR = PROJECT_ROOT / "data" / "processed" / "ideal"
REALISTIC_DIR = PROJECT_ROOT / "data" / "processed" / "realistic"

REALISTIC_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SHARED_DIR:", SHARED_DIR)
print("IDEAL_DIR:", IDEAL_DIR)
print("REALISTIC_DIR:", REALISTIC_DIR)

PROJECT_ROOT: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project
SHARED_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/shared
IDEAL_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/ideal
REALISTIC_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/realistic


In [4]:
required_files = [
    SHARED_DIR / "clean_childcare_facilities.csv",
    SHARED_DIR / "facility_geo_ready.csv",
    SHARED_DIR / "potential_locations_geo_ready.csv",
    IDEAL_DIR / "zipcode_demand_supply_ideal.csv",
    IDEAL_DIR / "build_options_ideal.csv",
]

missing_files = [str(f) for f in required_files if not f.exists()]

print("Required input files check:")
for f in required_files:
    print(f"  {f.name}: {f.exists()}")

if missing_files:
    raise FileNotFoundError(
        "The following required files are missing. Run the shared cleaning and ideal parameter notebooks first:\n"
        + "\n".join(missing_files)
    )

Required input files check:
  clean_childcare_facilities.csv: True
  facility_geo_ready.csv: True
  potential_locations_geo_ready.csv: True
  zipcode_demand_supply_ideal.csv: True
  build_options_ideal.csv: True


In [5]:
fac_clean = pd.read_csv(SHARED_DIR / "clean_childcare_facilities.csv")
facility_geo = pd.read_csv(SHARED_DIR / "facility_geo_ready.csv")
candidate_geo = pd.read_csv(SHARED_DIR / "potential_locations_geo_ready.csv")

zip_df_ideal = pd.read_csv(IDEAL_DIR / "zipcode_demand_supply_ideal.csv")
build_options = pd.read_csv(IDEAL_DIR / "build_options_ideal.csv")

zip_df = zip_df_ideal.copy()
zip_df["zipcode"] = zip_df["zipcode"].astype(str).str.zfill(5)

for col in [
    "pop_0_5",
    "child_pop_0_12",
    "req_total",
    "req_under5",
    "existing_total",
    "existing_under5",
    "total_threshold_rule",
]:
    if col in zip_df.columns:
        zip_df[col] = pd.to_numeric(zip_df[col], errors="coerce").fillna(0)

zip_df["req_total_original"] = zip_df["req_total"].astype(int)
zip_df["req_under5_original"] = zip_df["req_under5"].astype(int)

zip_df["req_total"] = np.where(
    zip_df["child_pop_0_12"] <= 0,
    0,
    zip_df["req_total_original"],
).astype(int)
zip_df["req_under5"] = np.where(
    zip_df["pop_0_5"] <= 0,
    0,
    zip_df["req_under5_original"],
).astype(int)

zip_df["zero_child_population_adjustment"] = (
    (zip_df["req_total"] != zip_df["req_total_original"])
    | (zip_df["req_under5"] != zip_df["req_under5_original"])
).astype(int)

zip_df["gap_total"] = np.maximum(0, zip_df["req_total"] - zip_df["existing_total"])
zip_df["gap_under5"] = np.maximum(0, zip_df["req_under5"] - zip_df["existing_under5"])

if "total_threshold_rule" in zip_df.columns:
    zip_df["is_desert_total_before"] = (
        (zip_df["req_total"] > 0)
        & (zip_df["existing_total"] <= zip_df["total_threshold_rule"])
    ).astype(int)

zip_df["is_under5_short_before"] = (
    zip_df["existing_under5"] < zip_df["req_under5"]
).astype(int)
zip_df["needs_intervention_before"] = (
    (zip_df["gap_total"] > 0) | (zip_df["gap_under5"] > 0)
).astype(int)

print(
    "Realistic demand rows adjusted for zero child population:",
    int(zip_df["zero_child_population_adjustment"].sum()),
)
display(zip_df.head())


Realistic demand rows adjusted for zero child population: 131


,zipcode,pop_0_5,pop_6_12,child_pop_0_12,employment_rate,average_income,employment_missing_flag,income_missing_flag,pop_0_5_missing_flag,pop_6_12_missing_flag,child_pop_0_12_missing_flag,high_demand,total_threshold_rule,req_total,req_under5,existing_total,existing_under5,n_active_facilities,gap_total,gap_under5,is_desert_total_before,is_under5_short_before,needs_intervention_before,req_total_original,req_under5_original,zero_child_population_adjustment
0,10001,744.0,1255.0,1999.0,0.595097,102878.033603,0,0,0,0,0,0,666.333333,667,496,609.0,24.0,9.0,58.0,472.0,1,1,1,667,496,0
1,10002,2142.0,4645.0,6787.0,0.520662,59604.041165,0,0,0,0,0,1,3393.500000,3394,1428,4724.0,198.0,56.0,0.0,1230.0,0,1,1,3394,1428,0
2,10003,1440.0,1510.0,2950.0,0.497244,114273.049645,0,0,0,0,0,0,983.333333,984,960,1995.0,0.0,7.0,0.0,960.0,0,1,1,984,960,0
3,10004,433.0,262.0,695.0,0.506661,132004.310345,0,0,0,0,0,0,231.666667,232,289,263.0,0.0,2.0,0.0,289.0,0,1,1,232,289,0
4,10005,484.0,318.0,802.0,0.665833,121437.713311,0,0,0,0,0,1,401.000000,402,323,39.0,0.0,1.0,363.0,323.0,1,1,1,402,323,0


In [6]:
for df in [fac_clean, facility_geo, candidate_geo, zip_df]:
    if "zipcode" in df.columns:
        df["zipcode"] = df["zipcode"].astype(str).str.zfill(5)

if "candidate_id" in candidate_geo.columns:
    candidate_geo["candidate_id"] = pd.to_numeric(candidate_geo["candidate_id"], errors="coerce").astype("Int64")

if "facility_id" in facility_geo.columns:
    facility_geo["facility_id"] = facility_geo["facility_id"].astype(str)

print("fac_clean columns:")
print(fac_clean.columns.tolist())
print("\nfacility_geo columns:")
print(facility_geo.columns.tolist())
print("\ncandidate_geo columns:")
print(candidate_geo.columns.tolist())
print("\nzip_df columns:")
print(zip_df.columns.tolist())
print("\nbuild_options columns:")
print(build_options.columns.tolist())

fac_clean columns:
['facility_id', 'program_type', 'facility_status', 'facility_name', 'city', 'zipcode', 'school_district_name', 'infant_capacity', 'toddler_capacity', 'preschool_capacity', 'school_age_capacity', 'children_capacity', 'total_capacity', 'latitude', 'longitude', 'is_active_facility', 'geo_missing_flag', 'under5_capacity_current', 'capacity_inconsistency_flag', 'zero_total_capacity_flag']

facility_geo columns:
['facility_id', 'facility_name', 'program_type', 'facility_status', 'is_active_facility', 'zipcode', 'latitude', 'longitude', 'geo_missing_flag', 'infant_capacity', 'toddler_capacity', 'preschool_capacity', 'school_age_capacity', 'children_capacity', 'under5_capacity_current', 'total_capacity']

candidate_geo columns:
['candidate_id', 'zipcode', 'latitude', 'longitude', 'geo_missing_flag']

zip_df columns:
['zipcode', 'pop_0_5', 'pop_6_12', 'child_pop_0_12', 'employment_rate', 'average_income', 'employment_missing_flag', 'income_missing_flag', 'pop_0_5_missing_flag

In [7]:
fac_existing = fac_clean.copy()

if "is_active_facility" not in fac_existing.columns:
    raise KeyError("Expected column 'is_active_facility' in clean_childcare_facilities.csv.")

fac_existing = fac_existing[fac_existing["is_active_facility"] == 1].copy()

for col in ["facility_id", "zipcode", "total_capacity", "under5_capacity_current"]:
    if col not in fac_existing.columns:
        raise KeyError(f"Expected column '{col}' in clean_childcare_facilities.csv.")

fac_existing["facility_id"] = fac_existing["facility_id"].astype(str)
fac_existing["zipcode"] = fac_existing["zipcode"].astype(str).str.zfill(5)

fac_existing["n_f"] = pd.to_numeric(fac_existing["total_capacity"], errors="coerce").fillna(0)
fac_existing["b_f"] = pd.to_numeric(fac_existing["under5_capacity_current"], errors="coerce").fillna(0)

fac_existing["b_f"] = np.minimum(fac_existing["b_f"], fac_existing["n_f"])
fac_existing = fac_existing[fac_existing["n_f"] > 0].copy()

print("Active facilities with positive capacity:", fac_existing.shape)
display(fac_existing[["facility_id", "zipcode", "n_f", "b_f"]].head())

Active facilities with positive capacity: (7712, 22)


,facility_id,zipcode,n_f,b_f
0,51349,10474,6,6
1,73544,10017,60,0
2,108407,11225,16,12
3,111953,11226,16,12
4,126425,10465,12,12


In [8]:
facility_geo["facility_id"] = facility_geo["facility_id"].astype(str)

geo_cols = ["facility_id", "facility_name", "program_type", "latitude", "longitude", "zipcode"]
missing_geo_cols = [c for c in geo_cols if c not in facility_geo.columns]
if missing_geo_cols:
    raise KeyError(f"Missing columns in facility_geo_ready.csv: {missing_geo_cols}")

fac_existing = fac_existing.merge(
    facility_geo[geo_cols],
    on="facility_id",
    how="left",
    suffixes=("", "_geo")
)

if "zipcode_geo" in fac_existing.columns:
    fac_existing.drop(columns=["zipcode_geo"], inplace=True)

print("Facilities missing latitude/longitude:", fac_existing[["latitude", "longitude"]].isna().any(axis=1).sum())
display(fac_existing.head())

Facilities missing latitude/longitude: 169


,facility_id,program_type,facility_status,facility_name,city,zipcode,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude,is_active_facility,geo_missing_flag,under5_capacity_current,capacity_inconsistency_flag,zero_total_capacity_flag,n_f,b_f,facility_name_geo,program_type_geo,latitude_geo,longitude_geo
0,51349,FDC,Registration,"Valerio, Gladys",Bronx,10474,Bronx 8,0,0,0,0,6,6,40.818399,-73.888816,1,False,6,0,0,6,6,"Valerio, Gladys",FDC,40.818399,-73.888816
1,73544,SACC,Registration,YMCA OF GREATER NY,New York,10017,Manhattan 2,0,0,0,60,0,60,40.753216,-73.970794,1,False,0,0,0,60,0,YMCA OF GREATER NY,SACC,40.753216,-73.970794
2,108407,GFDC,License,"Almond Tree Group Family Day Care, LLC",Brooklyn,11225,Brooklyn 17,0,0,0,4,12,16,NaN,NaN,1,True,12,0,0,16,12,"Almond Tree Group Family Day Care, LLC",GFDC,NaN,NaN
3,111953,GFDC,License,"Williams, Petal",Brooklyn,11226,Brooklyn 22,0,0,0,4,12,16,NaN,NaN,1,True,12,0,0,16,12,"Williams, Petal",GFDC,NaN,NaN
4,126425,GFDC,License,"Hernandez, Lidia",Bronx,10465,Bronx 8,0,0,0,0,12,12,40.841228,-73.823572,1,False,12,0,0,12,12,"Hernandez, Lidia",GFDC,40.841228,-73.823572


## Realistic expansion parameter design

For the realistic scenario, expansion is capped at **20%** of current facility capacity.
The code stores both:

- regime widths that partition integer expansions across `0-10%`, `10-15%`, and `15-20%`
- regime lower/upper bounds so the optimization model can apply the assignment's
  regime-based cost formula to the full expansion amount

This avoids understating capacity by flooring each block separately and keeps the Part 2
cost logic distinct from the Part 1 approximation.


In [9]:
cap_10 = np.floor(0.10 * fac_existing["n_f"]).astype(int)
cap_15 = np.floor(0.15 * fac_existing["n_f"]).astype(int)
cap_20 = np.floor(0.20 * fac_existing["n_f"]).astype(int)

fac_existing["cap1"] = cap_10
fac_existing["cap2"] = np.maximum(0, cap_15 - cap_10).astype(int)
fac_existing["cap3"] = np.maximum(0, cap_20 - cap_15).astype(int)
fac_existing["U_f_realistic"] = cap_20

fac_existing["regime1_lb"] = np.where(fac_existing["cap1"] > 0, 1, 0).astype(int)
fac_existing["regime1_ub"] = fac_existing["cap1"].astype(int)
fac_existing["regime2_lb"] = np.where(fac_existing["cap2"] > 0, cap_10 + 1, 0).astype(int)
fac_existing["regime2_ub"] = (fac_existing["cap1"] + fac_existing["cap2"]).astype(int)
fac_existing["regime3_lb"] = np.where(
    fac_existing["cap3"] > 0,
    fac_existing["cap1"] + fac_existing["cap2"] + 1,
    0,
).astype(int)
fac_existing["regime3_ub"] = fac_existing["U_f_realistic"].astype(int)

fac_existing["coef1"] = (20000 + 200 * fac_existing["n_f"]) / fac_existing["n_f"]
fac_existing["coef2"] = (20000 + 400 * fac_existing["n_f"]) / fac_existing["n_f"]
fac_existing["coef3"] = (20000 + 1000 * fac_existing["n_f"]) / fac_existing["n_f"]
fac_existing["missing_geo_flag"] = fac_existing[["latitude", "longitude"]].isna().any(axis=1).astype(int)

facility_expansion_params_realistic = fac_existing[
    [
        "facility_id",
        "facility_name",
        "program_type",
        "zipcode",
        "latitude",
        "longitude",
        "missing_geo_flag",
        "n_f",
        "b_f",
        "cap1",
        "cap2",
        "cap3",
        "U_f_realistic",
        "regime1_lb",
        "regime1_ub",
        "regime2_lb",
        "regime2_ub",
        "regime3_lb",
        "regime3_ub",
        "coef1",
        "coef2",
        "coef3",
    ]
].copy()

display(facility_expansion_params_realistic.head())


,facility_id,facility_name,program_type,zipcode,latitude,longitude,missing_geo_flag,n_f,b_f,cap1,cap2,cap3,U_f_realistic,regime1_lb,regime1_ub,regime2_lb,regime2_ub,regime3_lb,regime3_ub,coef1,coef2,coef3
0,51349,"Valerio, Gladys",FDC,10474,40.818399,-73.888816,0,6,6,0,0,1,1,0,0,0,0,1,1,3533.333333,3733.333333,4333.333333
1,73544,YMCA OF GREATER NY,SACC,10017,40.753216,-73.970794,0,60,0,6,3,3,12,1,6,7,9,10,12,533.333333,733.333333,1333.333333
2,108407,"Almond Tree Group Family Day Care, LLC",GFDC,11225,NaN,NaN,1,16,12,1,1,1,3,1,1,2,2,3,3,1450.000000,1650.000000,2250.000000
3,111953,"Williams, Petal",GFDC,11226,NaN,NaN,1,16,12,1,1,1,3,1,1,2,2,3,3,1450.000000,1650.000000,2250.000000
4,126425,"Hernandez, Lidia",GFDC,10465,40.841228,-73.823572,0,12,12,1,0,1,2,1,1,0,1,2,2,1866.666667,2066.666667,2666.666667


In [10]:
print("Any negative block capacities?", (facility_expansion_params_realistic[["cap1", "cap2", "cap3"]] < 0).any().any())
print("Any missing cost coefficients?", facility_expansion_params_realistic[["coef1", "coef2", "coef3"]].isna().any().any())
print("Total citywide realistic expansion capacity:", facility_expansion_params_realistic["U_f_realistic"].sum())

Any negative block capacities? False
Any missing cost coefficients? False
Total citywide realistic expansion capacity: 61197


## Candidate-site build options

The realistic model keeps the same new-facility size and cost menu from the ideal model. The only difference is that each new facility must be assigned to an actual candidate site.

In [11]:
candidate_required_cols = ["candidate_id", "zipcode", "latitude", "longitude"]
missing_candidate_cols = [c for c in candidate_required_cols if c not in candidate_geo.columns]
if missing_candidate_cols:
    raise KeyError(f"Missing columns in potential_locations_geo_ready.csv: {missing_candidate_cols}")

candidate_geo = candidate_geo.copy()
candidate_geo["zipcode"] = candidate_geo["zipcode"].astype(str).str.zfill(5)
candidate_geo["candidate_id"] = pd.to_numeric(candidate_geo["candidate_id"], errors="coerce").astype("Int64")

build_required_cols = ["size", "new_total_capacity", "new_under5_capacity_max", "fixed_build_cost"]
missing_build_cols = [c for c in build_required_cols if c not in build_options.columns]
if missing_build_cols:
    raise KeyError(f"Missing columns in build_options_ideal.csv: {missing_build_cols}")

candidate_build_options_realistic = candidate_geo.merge(build_options, how="cross")
display(candidate_build_options_realistic.head())
print("Candidate-site × build-size rows:", candidate_build_options_realistic.shape[0])

,candidate_id,zipcode,latitude,longitude,geo_missing_flag,size,new_total_capacity,new_under5_capacity_max,fixed_build_cost
0,1,10001,40.741893,-74.000140,False,small,100,50,65000
1,1,10001,40.741893,-74.000140,False,medium,200,100,95000
2,1,10001,40.741893,-74.000140,False,large,400,200,115000
3,2,10001,40.752007,-74.005436,False,small,100,50,65000
4,2,10001,40.752007,-74.005436,False,medium,200,100,95000


Candidate-site × build-size rows: 93300


In [12]:
def haversine_miles(lat1, lon1, lat2, lon2):
    """Compute great-circle distance in miles."""
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan

    radius_miles = 3958.7613

    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    return radius_miles * c

MIN_DIST = 0.06

## Distance conflict tables

The realistic model uses the minimum-distance rule only for decisions involving candidate sites:

- **candidate-existing**: if a candidate site is too close to an existing facility, that candidate site is blocked.
- **candidate-candidate**: if two candidate sites are too close, they cannot both be selected.
- **existing-existing**: these pairs are reported for diagnostics only and are not enforced in the model, because existing facilities cannot be relocated.

In [13]:
cand_exist_pairs = candidate_geo.merge(
    facility_expansion_params_realistic[["facility_id", "zipcode", "latitude", "longitude"]],
    on="zipcode",
    how="inner",
    suffixes=("_cand", "_exist")
)

cand_exist_pairs["distance_miles"] = cand_exist_pairs.apply(
    lambda r: haversine_miles(
        r["latitude_cand"], r["longitude_cand"],
        r["latitude_exist"], r["longitude_exist"]
    ),
    axis=1
)

candidate_existing_conflicts = cand_exist_pairs[
    cand_exist_pairs["distance_miles"] < MIN_DIST
].copy()

candidate_existing_conflicts = candidate_existing_conflicts[
    ["candidate_id", "facility_id", "zipcode", "distance_miles"]
].drop_duplicates()

print("Candidate-existing conflicts:", candidate_existing_conflicts.shape)
display(candidate_existing_conflicts.head())

Candidate-existing conflicts: (4730, 4)


,candidate_id,facility_id,zipcode,distance_miles
36,5,837597,10001,0.026513
41,5,229433,10001,0.026513
423,48,837597,10001,0.021816
428,48,229433,10001,0.021816
585,66,837597,10001,0.043996


In [14]:
cand1 = candidate_geo.rename(columns={
    "candidate_id": "candidate_id_1",
    "latitude": "latitude_1",
    "longitude": "longitude_1"
})
cand2 = candidate_geo.rename(columns={
    "candidate_id": "candidate_id_2",
    "latitude": "latitude_2",
    "longitude": "longitude_2"
})

candidate_candidate_pairs = cand1.merge(cand2, on="zipcode", how="inner")
candidate_candidate_pairs = candidate_candidate_pairs[
    candidate_candidate_pairs["candidate_id_1"] < candidate_candidate_pairs["candidate_id_2"]
].copy()

candidate_candidate_pairs["distance_miles"] = candidate_candidate_pairs.apply(
    lambda r: haversine_miles(
        r["latitude_1"], r["longitude_1"],
        r["latitude_2"], r["longitude_2"]
    ),
    axis=1
)

candidate_candidate_conflicts = candidate_candidate_pairs[
    candidate_candidate_pairs["distance_miles"] < MIN_DIST
].copy()

candidate_candidate_conflicts = candidate_candidate_conflicts[
    ["candidate_id_1", "candidate_id_2", "zipcode", "distance_miles"]
].drop_duplicates()

print("Candidate-candidate conflicts:", candidate_candidate_conflicts.shape)
display(candidate_candidate_conflicts.head())

Candidate-candidate conflicts: (11455, 4)


,candidate_id_1,candidate_id_2,zipcode,distance_miles
446,5,47,10001,0.047333
447,5,48,10001,0.013688
465,5,66,10001,0.020589
581,6,82,10001,0.016552
644,7,45,10001,0.035234


In [15]:
exist1 = facility_expansion_params_realistic.rename(columns={
    "facility_id": "facility_id_1",
    "latitude": "latitude_1",
    "longitude": "longitude_1"
})
exist2 = facility_expansion_params_realistic.rename(columns={
    "facility_id": "facility_id_2",
    "latitude": "latitude_2",
    "longitude": "longitude_2"
})

existing_existing_pairs = exist1.merge(exist2, on="zipcode", how="inner")
existing_existing_pairs = existing_existing_pairs[
    existing_existing_pairs["facility_id_1"] < existing_existing_pairs["facility_id_2"]
].copy()

existing_existing_pairs["distance_miles"] = existing_existing_pairs.apply(
    lambda r: haversine_miles(
        r["latitude_1"], r["longitude_1"],
        r["latitude_2"], r["longitude_2"]
    ),
    axis=1
)

existing_existing_too_close = existing_existing_pairs[
    existing_existing_pairs["distance_miles"] < MIN_DIST
].copy()

existing_existing_too_close = existing_existing_too_close[
    ["facility_id_1", "facility_id_2", "zipcode", "distance_miles"]
].drop_duplicates()

print("Existing-existing pairs below threshold (diagnostic only):", existing_existing_too_close.shape)
display(existing_existing_too_close.head())

Existing-existing pairs below threshold (diagnostic only): (7644, 4)


,facility_id_1,facility_id_2,zipcode,distance_miles
5,51349,609919,10474,0.027891
11,51349,633751,10474,0.027891
13,51349,601505,10474,0.042375
411,274852,385789,10460,0.055933
467,274852,743579,10460,0.037859


In [16]:
blocked_candidates = (
    candidate_existing_conflicts[["candidate_id"]]
    .drop_duplicates()
    .assign(blocked_by_existing=1)
)

candidate_geo_realistic = candidate_geo.merge(
    blocked_candidates,
    on="candidate_id",
    how="left"
)

candidate_geo_realistic["blocked_by_existing"] = candidate_geo_realistic["blocked_by_existing"].fillna(0).astype(int)

print(candidate_geo_realistic["blocked_by_existing"].value_counts(dropna=False))
display(candidate_geo_realistic.head())

blocked_by_existing
0    28522
1     2578
Name: count, dtype: int64


,candidate_id,zipcode,latitude,longitude,geo_missing_flag,blocked_by_existing
0,1,10001,40.741893,-74.000140,False,0
1,2,10001,40.752007,-74.005436,False,0
2,3,10001,40.750545,-73.997147,False,0
3,4,10001,40.744080,-74.001932,False,0
4,5,10001,40.748690,-73.999341,False,1


In [17]:
candidate_build_options_realistic = candidate_build_options_realistic.merge(
    candidate_geo_realistic[["candidate_id", "blocked_by_existing"]],
    on="candidate_id",
    how="left"
)

candidate_build_options_realistic["blocked_by_existing"] = (
    candidate_build_options_realistic["blocked_by_existing"].fillna(0).astype(int)
)

candidate_build_options_feasible = candidate_build_options_realistic[
    candidate_build_options_realistic["blocked_by_existing"] == 0
].copy()

print("All candidate-size options:", candidate_build_options_realistic.shape)
print("Feasible candidate-size options:", candidate_build_options_feasible.shape)

All candidate-size options: (93300, 10)
Feasible candidate-size options: (85566, 10)


In [18]:
realistic_parameter_summary = pd.DataFrame({
    "metric": [
        "n_zipcodes",
        "n_zipcodes_adjusted_for_zero_population",
        "n_existing_facilities_expandable",
        "citywide_existing_total_capacity",
        "citywide_existing_under5_capacity",
        "citywide_realistic_expansion_capacity",
        "n_existing_facilities_missing_geo",
        "n_candidate_sites",
        "n_blocked_candidate_sites",
        "n_feasible_candidate_sites",
        "n_candidate_candidate_conflicts",
        "n_candidate_existing_conflicts",
        "n_existing_existing_too_close_diagnostic",
    ],
    "value": [
        zip_df["zipcode"].nunique(),
        int(zip_df["zero_child_population_adjustment"].sum()),
        facility_expansion_params_realistic["facility_id"].nunique(),
        facility_expansion_params_realistic["n_f"].sum(),
        facility_expansion_params_realistic["b_f"].sum(),
        facility_expansion_params_realistic["U_f_realistic"].sum(),
        int(facility_expansion_params_realistic["missing_geo_flag"].sum()),
        candidate_geo_realistic["candidate_id"].nunique(),
        candidate_geo_realistic["blocked_by_existing"].sum(),
        (candidate_geo_realistic["blocked_by_existing"] == 0).sum(),
        len(candidate_candidate_conflicts),
        len(candidate_existing_conflicts),
        len(existing_existing_too_close),
    ]
})

display(realistic_parameter_summary)


,metric,value
0,n_zipcodes,311
1,n_zipcodes_adjusted_for_zero_population,131
2,n_existing_facilities_expandable,7712
3,citywide_existing_total_capacity,317501
4,citywide_existing_under5_capacity,69153
5,citywide_realistic_expansion_capacity,61197
6,n_existing_facilities_missing_geo,169
7,n_candidate_sites,31100
8,n_blocked_candidate_sites,2578
9,n_feasible_candidate_sites,28522


In [19]:
facility_expansion_params_realistic.to_csv(
    REALISTIC_DIR / "facility_expansion_params_realistic.csv",
    index=False
)

zip_df.to_csv(
    REALISTIC_DIR / "zipcode_demand_supply_realistic.csv",
    index=False
)

candidate_geo_realistic.to_csv(
    REALISTIC_DIR / "candidate_sites_realistic.csv",
    index=False
)

candidate_build_options_feasible.to_csv(
    REALISTIC_DIR / "candidate_build_options_realistic.csv",
    index=False
)

candidate_candidate_conflicts.to_csv(
    REALISTIC_DIR / "candidate_candidate_conflicts.csv",
    index=False
)

candidate_existing_conflicts.to_csv(
    REALISTIC_DIR / "candidate_existing_conflicts.csv",
    index=False
)

existing_existing_too_close.to_csv(
    REALISTIC_DIR / "existing_existing_too_close_diagnostic.csv",
    index=False
)

realistic_parameter_summary.to_csv(
    REALISTIC_DIR / "realistic_parameter_summary.csv",
    index=False
)

print("Realistic parameter files saved to:", REALISTIC_DIR)


Realistic parameter files saved to: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/realistic


In [20]:
REALISTIC_ASSUMPTIONS = {
    "distance_threshold_miles": 0.06,
    "distance_enforced_within_zipcode_only": True,
    "existing_existing_conflicts_are_diagnostic_only": True,
    "candidate_existing_conflict_blocks_candidate": True,
    "use_under5_capacity_current_for_existing_capacity": True,
    "new_build_costs_and_capacities_reused_from_ideal_model": True,
    "realistic_demand_adjusts_zero_child_population_rows": True,
    "realistic_total_cap_rule": "x_f <= floor(0.20 * n_f)",
    "realistic_piecewise_cost_rule": "assignment regime-based total expansion cost",
    "expansion_piecewise_regimes": [
        "1_to_floor(0.10*n_f)",
        "floor(0.10*n_f)+1_to_floor(0.15*n_f)",
        "floor(0.15*n_f)+1_to_floor(0.20*n_f)",
    ],
    "existing_facilities_with_missing_geo_are_not_blocking_candidates": True,
}

with open(REALISTIC_DIR / "assumptions_realistic.json", "w", encoding="utf-8") as f:
    json.dump(REALISTIC_ASSUMPTIONS, f, ensure_ascii=False, indent=2)

pd.DataFrame({
    "assumption": list(REALISTIC_ASSUMPTIONS.keys()),
    "value": list(REALISTIC_ASSUMPTIONS.values())
}).to_csv(REALISTIC_DIR / "assumptions_realistic.csv", index=False)

print("Saved assumptions_realistic.json and assumptions_realistic.csv")


Saved assumptions_realistic.json and assumptions_realistic.csv
